<a href="https://colab.research.google.com/github/Diego-Rodriguez16/AlgorithmsUN2024II/blob/main/4_Programacion_genetica/Taller_4_Programacion_Genetica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación Genética – Ejercicios 1 y 3
- **Ejercicio 1:** circuito lógico (codificador de 7 segmentos) con PG.
- **Ejercicio 3:** detección de fraudes con PG.

Librería usada: `gplearn` (programación genética en Python).

In [ ]:
!pip install gplearn

## Ejercicio 1 – Codificador de 7 segmentos

**Problema:** convertir un dígito (0-9) en 4 bits (A, B, C, D) a los 7 segmentos (a-g) de un display.

**Conjunto de terminales:** las 4 entradas binarias `A, B, C, D` (los bits del dígito).

**Conjunto de funciones:** `AND`, `OR`, `XOR`, `NOT`.

**Función de aptitud:** error absoluto medio entre la salida del circuito y la tabla de verdad (mientras más cerca de 0, mejor). Se evolucionó un circuito por cada segmento.

In [ ]:
import numpy as np
from gplearn.genetic import SymbolicRegressor
from gplearn.functions import make_function

# Funciones lógicas (trabajan con 0 y 1)
f_and = make_function(lambda a, b: np.minimum(a, b), 'and_', 2)
f_or  = make_function(lambda a, b: np.maximum(a, b), 'or_', 2)
f_xor = make_function(lambda a, b: np.abs(a - b), 'xor_', 2)
f_not = make_function(lambda a: 1 - a, 'not_', 1)

# Entradas: dígitos 0-9 en 4 bits (A B C D)
X = np.array([[(d >> 3) & 1, (d >> 2) & 1, (d >> 1) & 1, d & 1] for d in range(10)])

# Salidas: segmentos a b c d e f g para cada dígito
tabla = ["1111110", "0110000", "1101101", "1111001", "0110011",
         "1011011", "1011111", "1110000", "1111111", "1111011"]
Y = np.array([[int(c) for c in fila] for fila in tabla])

In [ ]:
for i, seg in enumerate("abcdefg"):
    gp = SymbolicRegressor(population_size=1000, generations=30,
                           function_set=(f_and, f_or, f_xor, f_not),
                           metric='mean absolute error',
                           const_range=None, parsimony_coefficient=0.01,
                           random_state=0, verbose=0)
    gp.fit(X, Y[:, i])
    pred = np.clip(np.round(gp.predict(X)), 0, 1)
    aciertos = int((pred == Y[:, i]).sum())
    print(f"Segmento {seg}: {aciertos}/10 correctos -> {gp._program}")
    # X0=A, X1=B, X2=C, X3=D

## Ejercicio 3 – Detección de fraudes

Se genera un archivo aleatorio con: **monto**, **hora del día**, **distancia a la residencia**, **ingreso mensual** y **fraude** (sí=1 / no=0).
Para que sea más realista, el fraude es más probable cuando el monto es alto respecto al ingreso, la compra es de madrugada o ocurre lejos de casa.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)
n = 2000
df = pd.DataFrame({
    "monto": np.random.exponential(300000, n).round(),            # pesos
    "hora": np.random.randint(0, 24, n),                          # 0-23
    "distancia": np.random.exponential(10, n).round(1),           # km
    "ingreso": np.random.normal(3000000, 800000, n).clip(1000000).round(),
})

# Regla "realista" + algo de ruido
riesgo = df.monto / df.ingreso * 3 + (df.hora < 6) * 1.0 + (df.distancia > 30) * 1.5
df["fraude"] = (riesgo + np.random.normal(0, 0.3, n) > 2.2).astype(int)

df.to_csv("fraudes.csv", index=False)
print("Porcentaje de fraudes:", df.fraude.mean())
df.head()

**Terminales:** `monto`, `hora`, `distancia`, `ingreso` (+ constantes aleatorias).
**Funciones:** `+, -, *, /`.
**Aptitud:** error de clasificación (log-loss) sobre los datos de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from gplearn.genetic import SymbolicClassifier

X = df[["monto", "hora", "distancia", "ingreso"]]
y = df["fraude"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

gp = SymbolicClassifier(population_size=500, generations=15,
                        function_set=('add', 'sub', 'mul', 'div'),
                        parsimony_coefficient=0.001, random_state=0, verbose=1)
gp.fit(X_tr, y_tr)

pred = gp.predict(X_te)
print("Exactitud:", accuracy_score(y_te, pred))
print("Matriz de confusión:\n", confusion_matrix(y_te, pred))
print("Modelo encontrado:", gp._program)  # X0=monto, X1=hora, X2=distancia, X3=ingreso